# Entrenar una red neuronal ANN(2, b, 1)

Genera puntos con ruido sobre una «colina» $y = e^{-r^2/10}$ y entrena una red de 2 entradas,
`b` neuronas ocultas y 1 salida con descenso de gradiente. Se ve cómo la superficie de la red
se ajusta a los puntos y cómo baja el error (MSE) época a época.

Usa numpy matplotlib ipywidgets (todas en `requirements.txt`).

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
from ipywidgets import HTML, Button, FloatSlider, HBox, IntSlider, Output, VBox


# ==========================================
# 1. MATEMÁTICAS DE LA RED NEURONAL
# ==========================================
def sigmoid(x):
    """Función sigmoide, 1 / (1 + e^-x)."""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


def sigmoid_deriv(x):
    """Derivada de la sigmoide."""
    s = sigmoid(x)
    return s * (1 - s)


class ANNState:
    """Datos de entrenamiento, pesos de la red e historial del error."""

    def __init__(self):
        self.X = None
        self.Y = None
        self.W1, self.b1 = None, None
        self.W2, self.b2 = None, None
        self.loss_history = []


state = ANNState()

# ==========================================
# 2. INTERFAZ DE USUARIO (WIDGETS)
# ==========================================
HTML_header = HTML(
    "<h2 style='color:#2c3e50;'>Entrenamiento de ANN(2, b, 1) con Gradiente Descendente</h2>"
)

slider_b = IntSlider(value=4, min=1, max=10, description="Ocultas (b):")
slider_lr = FloatSlider(
    value=0.5, min=0.01, max=2.0, step=0.01, description="Tasa Aprend. (LR):"
)
slider_epochs = IntSlider(
    value=500, min=100, max=10000, step=100, description="Épocas:"
)

btn_generate = Button(
    description="🎲 Generar Datos", button_style="info", icon="refresh"
)
btn_train = Button(description="🚀 Entrenar ANN", button_style="success", icon="play")

controls_box = HBox(
    [VBox([slider_b, slider_lr, slider_epochs]), VBox([btn_generate, btn_train])]
)
out_plot = Output()


# ==========================================
# 3. FUNCIONES DE LÓGICA Y DIBUJO
# ==========================================
def generate_data(b):
    """Genera una superficie 3D con ruido y reinicia los pesos de la red"""
    # Generar 100 puntos (x1, x2) aleatorios entre -5 y 5
    state.X = np.random.uniform(-5, 5, (150, 2))

    # Función objetivo: una "colina" 3D normalizada entre 0 y 1 + ruido
    R = np.sqrt(state.X[:, 0] ** 2 + state.X[:, 1] ** 2)
    y_true = np.exp(-(R**2) / 10)
    noise = np.random.normal(0, 0.05, 150)
    state.Y = np.clip(y_true + noise, 0, 1).reshape(-1, 1)

    # Inicializar Pesos (Distribución normal)
    b_nodes = slider_b.value
    state.W1 = np.random.randn(2, b_nodes)
    state.b1 = np.zeros((1, b_nodes))
    state.W2 = np.random.randn(b_nodes, 1)
    state.b2 = np.zeros((1, 1))
    state.loss_history = []

    draw_plots(epoch=0)


def draw_plots(epoch=0):
    """Dibuja la superficie 3D y la curva de pérdida"""
    with out_plot:
        clear_output(wait=True)
        fig = plt.figure(figsize=(15, 6))

        # --- GRÁFICA 1: SUPERFICIE 3D ---
        ax1 = fig.add_subplot(1, 2, 1, projection="3d")
        # Dibujar los puntos de datos reales
        ax1.scatter(
            state.X[:, 0],
            state.X[:, 1],
            state.Y[:, 0],
            color="black",
            s=15,
            label="Datos Reales (con ruido)",
        )

        # Crear una cuadrícula para predecir y dibujar la superficie de la ANN
        x_grid = np.linspace(-5, 5, 20)
        y_grid = np.linspace(-5, 5, 20)
        X1, X2 = np.meshgrid(x_grid, y_grid)
        grid_points = np.c_[X1.ravel(), X2.ravel()]

        # Forward pass para la cuadrícula (Predicción)
        Z1_grid = np.dot(grid_points, state.W1) + state.b1
        A1_grid = sigmoid(Z1_grid)
        Z2_grid = np.dot(A1_grid, state.W2) + state.b2
        A2_grid = sigmoid(Z2_grid).reshape(20, 20)

        ax1.plot_surface(X1, X2, A2_grid, cmap="coolwarm", alpha=0.7, edgecolor="none")
        ax1.set_title(f"Predicción ANN (Época: {epoch})", fontsize=14)
        ax1.set_xlabel("x1")
        ax1.set_ylabel("x2")
        ax1.set_zlabel("y")
        ax1.set_zlim(0, 1)
        ax1.legend()

        # --- GRÁFICA 2: FUNCIÓN DE PÉRDIDA (ERROR) ---
        ax2 = fig.add_subplot(1, 2, 2)
        if len(state.loss_history) > 0:
            ax2.plot(state.loss_history, color="#e74c3c", linewidth=2)
        ax2.set_title(
            f"Error Cuadrático Medio (MSE): {state.loss_history[-1]:.4f}"
            if state.loss_history
            else "Función de Pérdida (MSE)",
            fontsize=14,
        )
        ax2.set_xlabel("Épocas")
        ax2.set_ylabel("Error")
        ax2.set_xlim(0, max(10, slider_epochs.value))
        ax2.set_ylim(
            0, max(0.2, max(state.loss_history) if state.loss_history else 0.2)
        )
        ax2.grid(True, linestyle="--", alpha=0.6)

        plt.tight_layout()
        plt.show()


def train_network(b):
    """Ejecuta el bucle de Gradiente Descendente con animación"""
    if state.X is None:
        generate_data(None)

    lr = slider_lr.value
    epochs = slider_epochs.value

    # Bucle evolutivo de entrenamiento
    for epoch in range(1, epochs + 1):
        # 1. FORWARD PASS
        Z1 = np.dot(state.X, state.W1) + state.b1
        A1 = sigmoid(Z1)
        Z2 = np.dot(A1, state.W2) + state.b2
        A2 = sigmoid(Z2)

        # Calcular y guardar el Error Cuadrático Medio (MSE)
        loss = np.mean((A2 - state.Y) ** 2)
        state.loss_history.append(loss)

        # 2. BACKWARD PASS (Gradiente Descendente)
        m = state.X.shape[0]  # Número de datos

        dZ2 = (A2 - state.Y) * sigmoid_deriv(Z2)
        dW2 = np.dot(A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        dZ1 = np.dot(dZ2, state.W2.T) * sigmoid_deriv(Z1)
        dW1 = np.dot(state.X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        # 3. ACTUALIZAR PESOS
        state.W1 -= lr * dW1
        state.b1 -= lr * db1
        state.W2 -= lr * dW2
        state.b2 -= lr * db2

        # Actualizar la animación visual cada ciertas épocas (para no bloquear el ordenador)
        if epoch % max(1, (epochs // 20)) == 0 or epoch == epochs:
            draw_plots(epoch)
            time.sleep(0.01)


# ==========================================
# 4. EVENTOS Y EJECUCIÓN INICIAL
# ==========================================
btn_generate.on_click(generate_data)
btn_train.on_click(train_network)

# Mostrar Interfaz
display(HTML_header, controls_box, out_plot)

# Generar datos iniciales automáticamente
generate_data(None)

HTML(value="<h2 style='color:#2c3e50;'>Entrenamiento de ANN(2, b, 1) con Gradiente Descendente</h2>")

Output()